# E10d · Modelos de lenguaje de n-gramas por clase (Kneser-Ney)
**Parte de:** E8b (mejor modelo actual: *bagging* de 5 semillas con los 5 modelos + LightGBM, CV 0,8179).

**Hipótesis D.** Todos los modelos base actuales son discriminativos sobre TF-IDF (o léxico). Un **modelo generativo** mira el texto de otra forma: aprende *cómo se escribe* un tweet de odio frente a uno que no lo es, con el orden de caracteres y palabras. Si su error es distinto, podría dar diversidad al stacking.

**Qué se hace.** Para cada etiqueta (HS, TR, AG) se entrena un modelo de lenguaje (LM) con los tweets de la clase 1 y otro con los de la clase 0, en dos niveles:
- **caracteres**, orden máximo 6 (se usan las puntuaciones de los órdenes 3, 4, 5 y 6);
- **palabras**, orden máximo 3 (órdenes 1, 2 y 3; palabras con frecuencia < 2 pasan a `UNK`).

El suavizado es **Kneser-Ney modificado interpolado** (Chen y Goodman, 1999), implementado a mano con `Counter` en `ext_lm.py`: descuentos D1, D2, D3+ por orden estimados con los conteos de conteos; el orden superior usa frecuencias y los inferiores, número de contextos distintos. La idea de codificar cada palabra como un carácter Unicode (para tratar palabras y caracteres con el mismo código) se tomó de TweetAge v6.

**Variables.** Para cada tweet y cada orden: *log-likelihood ratio* (LLR) = log-verosimilitud media por símbolo bajo el LM de la clase 1 menos la de la clase 0. Son 7 LLR por etiqueta. Una regresión logística pequeña (`C=1`, `class_weight='balanced'`) las convierte en probabilidad.

**Sin fuga.** Los LM se ajustan solo con la partición de entrenamiento del fold. Las LLR de entrenamiento de la LR se obtienen con una **CV interna de 5 particiones** dentro de ese fold (cada tweet se puntúa con LMs que no lo vieron); si no, la LR aprendería sobre LLR "en muestra", mucho más extremas que las de validación. El test solo se puntúa.

**Modelos registrados** (en `ext_lm.py`):
| Nombre | Descripción |
|---|---|
| `LM-KN` | LLR de caracteres (órdenes 3-6) + palabras (1-3) → LR por etiqueta |
| `LM-KN-PS` | Powerset generativo: un LM por combinación HS-TR-AG (6 combinaciones), puntuaciones centradas → LR multinomial → marginales |
| `LM-KN+lex` | `LM-KN` con las 38 variables de léxico agregadas a la LR |
| `LM-KN-char`, `LM-KN-word` | Solo caracteres / solo palabras (registrados para pruebas; la comparación por nivel se hizo con la semilla 42, sección 2) |

### Glosario de esta prueba
| Término | Significado |
|---|---|
| **HS / TR / AG** | Las 3 etiquetas: discurso de odio / odio dirigido a una persona concreta / agresividad |
| **F1 medio** | F1 de la clase 1 de cada etiqueta, promediado entre HS, TR y AG (métrica principal) |
| **CV repetida** | Validación cruzada de 5 particiones repetida con 3 semillas (42, 43, 44); se reporta media ± desviación |
| **OOF** | *Out-of-fold*: predicción de cada fila del train hecha por un modelo que no la vio al entrenar |
| **Modelo de lenguaje (LM)** | Modelo que da la probabilidad de un texto, símbolo a símbolo: P(siguiente carácter o palabra \| los anteriores) |
| **n-grama / orden** | Secuencia de n símbolos. Un LM de orden n mira los n−1 símbolos anteriores |
| **Modelo generativo / discriminativo** | Generativo: modela cómo es el texto de cada clase, P(texto \| clase). Discriminativo: modela directamente P(clase \| texto), como la LR |
| **Suavizado** | Dar probabilidad a n-gramas no vistos quitando un poco de masa a los vistos |
| **Kneser-Ney (KN)** | Suavizado que descuenta una cantidad fija a cada conteo y reparte esa masa con un modelo de orden menor, que cuenta *en cuántos contextos distintos* aparece cada símbolo |
| **KN modificado** | Variante con tres descuentos (para conteos 1, 2 y 3+) estimados de los datos |
| **Log-verosimilitud media** | Suma de log P de cada símbolo dividida por el número de símbolos. Compara textos de distinta longitud |
| **LLR** | *Log-likelihood ratio*: log-verosimilitud bajo el LM de la clase 1 menos la de la clase 0. Positivo = el texto "se parece más" a la clase 1 |
| **CV interna** | CV dentro de la partición de entrenamiento del fold, para obtener variables de entrenamiento sin fuga |
| **Powerset** | Tratar la combinación HS-TR-AG como una sola clase (`000`, `101`, `111`...) |
| **Correlación de Pearson** | Mide de −1 a 1 cuánto se mueven juntas dos series; cerca de 1 = los modelos dicen casi lo mismo |
| **Stacking / *seed bagging*** | Un metamodelo combina los OOF de los modelos base / se promedian los OOF de 5 semillas de partición antes del stacking |

El glosario completo está en `RESUMEN.md`.

In [1]:
import os, sys, platform, re
sys.path.insert(0, os.path.abspath('.'))          # el notebook vive en experimentos/, junto a toxic_base.py
os.environ.setdefault('OMP_NUM_THREADS', '3')
import numpy as np, pandas as pd, sklearn, scipy
import toxic_base as tb
import ext_lm                                     # registra el bloque 'idx' y los modelos LM-KN*
pd.set_option('display.width', 220); pd.set_option('display.max_columns', 30); pd.set_option('display.max_colwidth', 120)
print('Python', platform.python_version(), '| numpy', np.__version__, '| pandas', pd.__version__,
      '| scikit-learn', sklearn.__version__, '| scipy', scipy.__version__)
print('Semillas de la CV repetida:', tb.SEEDS, '| particiones:', tb.N_FOLDS)
d = tb.Datos()
print('Datos en', d.data_dir, '| train', d.train.shape, '| test', d.test.shape)
print('Modelos LM registrados:', [m for m in tb.MODELOS if m.startswith('LM-')])

Python 3.12.3 | numpy 2.5.4 | pandas 3.0.6 | scikit-learn 1.9.1 | scipy 1.18.1
Semillas de la CV repetida: (42, 43, 44) | particiones: 5


Datos en /home/jc/repositories/natural-language-processing-workshops/workshops/Kaggle-tweet-toxic/tweet-toxic-2026-puj | train (1731, 5) | test (1154, 2)
Modelos LM registrados: ['LM-KN', 'LM-KN-char', 'LM-KN-word', 'LM-KN-PS', 'LM-KN+lex']


## 1. Qué aprende un LM por clase (LMs ajustados con todo el train, solo para inspección)
Se ajusta un LM de palabras de orden 2 para HS=1 y otro para HS=0 con todo el train. Las LLR de esta celda son "en muestra" (los LM vieron esos tweets), así que solo sirven para ver qué tipo de texto empuja hacia cada clase; no se usan en ningún modelo.

In [2]:
tx = list(d.TXT_TR.values)
enc, V = ext_lm._codificador(tx, 'word')
e = [enc(t) for t in tx]
lms = {c: ext_lm.KN(2, V).fit([s for s, y in zip(e, d.Y[:, 0]) if y == c]) for c in (0, 1)}
S = np.array([[lms[c].logprob(s)[1] for c in (0, 1)] for s in e])
llr = S[:, 1] - S[:, 0]
print('Vocabulario de palabras (frecuencia >= 2):', V - 2)
print('Perplejidad media por palabra (en muestra) | HS=1 bajo LM1: %.0f, bajo LM0: %.0f | HS=0 bajo LM1: %.0f, bajo LM0: %.0f' % tuple(
    np.exp(-S[d.Y[:, 0] == y, c].mean()) for y in (1, 0) for c in (1, 0)))
print('LLR media: HS=1 -> %.2f | HS=0 -> %.2f' % (llr[d.Y[:, 0] == 1].mean(), llr[d.Y[:, 0] == 0].mean()))
orden = np.argsort(llr)
pd.DataFrame({'LLR': np.r_[llr[orden[-5:]][::-1], llr[orden[:5]]].round(2),
              'HS real': np.r_[d.Y[orden[-5:][::-1], 0], d.Y[orden[:5], 0]],
              'texto normalizado': np.r_[d.TXT_TR.values[orden[-5:][::-1]], d.TXT_TR.values[orden[:5]]]},
             index=['más HS'] * 5 + ['menos HS'] * 5)

Vocabulario de palabras (frecuencia >= 2): 1998
Perplejidad media por palabra (en muestra) | HS=1 bajo LM1: 26, bajo LM0: 104 | HS=0 bajo LM1: 126, bajo LM0: 32
LLR media: HS=1 -> 1.40 | HS=0 -> -1.38


,LLR,HS real,texto normalizado
más HS,4.18,1,no me llames frijolero pinche gringo puetero
más HS,3.52,1,es aguacate malditos sudacas
más HS,3.31,1,xuser jajajaja callate zorra
más HS,3.31,1,[3/1 03:47] aa brii: boluda mira vos [3/1 03:48] aa brii: sos re zorra y casi no t nombran lo bueno de hacer las cos...
más HS,3.21,1,xuser noo callate perra que estoy buscando algo para poder dormir jajaj
menos HS,-4.82,0,xuser cllate la puta boca y feliz navidad amiga <3<3<3<3<3<3
menos HS,-3.56,0,si unf es acoso cibernetico todos estariamos presos dejen d decir estupideces ridiculas q son iheartawards bestboyba...
menos HS,-3.42,0,que pinche naco decirle estados unidos al gabacho
menos HS,-3.15,0,el guapsimo arabe quiere que le ensee espaol haha
menos HS,-3.15,0,anoche estaba hablando con mi beb ;v y cmo estaba casi dormida le escrib en rabe jajajajajaja


## 2. Validación cruzada repetida del modelo solo
Elección de órdenes (hecha antes con la semilla 42, 5 particiones; F1 medio de la LR sobre las LLR): solo caracteres 3-6 → 0,716; solo palabras 1-3 → 0,752; caracteres 3-6 + palabras 1-3 → **0,759**; los 9 órdenes → 0,759; usar sumas en vez de medias por símbolo → 0,753. Se fijó caracteres 3-6 + palabras 1-3. Los LMs de caracteres solos son claramente peores: con ~700 tweets por clase, la mayor parte de la probabilidad de un tweet la explica el español común, no la clase.

Las probabilidades OOF quedan en caché en `oof/base/` (cada semilla tarda unos 2 min por modelo).

In [3]:
R, OOF = {}, {}
for b in ['LR', 'LM-KN', 'LM-KN-PS', 'LM-KN+lex']:
    OOF[b] = {s: tb.oof(d, b, s) for s in tb.SEEDS}
    R[b] = tb.evaluar(d, OOF[b]); tb.log('OOF listo:', b)
tb.tabla(R)

[     1s] OOF listo: LR


[     1s] OOF listo: LM-KN


[     1s] OOF listo: LM-KN-PS


[     1s] OOF listo: LM-KN+lex


,F1 HS,F1 TR,F1 AG,F1 medio,±,EMR,F1 medio por semilla
Configuración,,,,,,,
LR,0.7924,0.8246,0.7899,0.8023,0.0020,0.7499,0.803 / 0.805 / 0.800
LM-KN,0.7621,0.7917,0.7214,0.7584,0.0010,0.6905,0.759 / 0.759 / 0.757
LM-KN-PS,0.7389,0.7740,0.6841,0.7323,0.0042,0.7113,0.737 / 0.727 / 0.734
LM-KN+lex,0.7959,0.8133,0.7896,0.7996,0.0026,0.7495,0.798 / 0.798 / 0.803


### Correlación de las probabilidades OOF con la LR
Si la correlación es baja, el modelo aporta información distinta; si es alta, el stacking ya tiene lo mismo.

In [4]:
corr = pd.DataFrame({b: [np.mean([np.corrcoef(OOF[b][s][:, j], OOF['LR'][s][:, j])[0, 1] for s in tb.SEEDS])
                          for j in range(3)] for b in ['LM-KN', 'LM-KN-PS', 'LM-KN+lex']}, index=tb.LABELS).T
corr['media'] = corr.mean(1)
for b in tb.BASES_ORIGINALES + ['LGBM']:
    if b != 'LR':
        corr.loc[f'(referencia) {b}'] = [np.mean([np.corrcoef(tb.oof(d, b, s)[:, j], OOF['LR'][s][:, j])[0, 1] for s in tb.SEEDS])
                                         for j in range(3)] + [np.nan]
corr['media'] = corr[tb.LABELS].mean(1)
corr.round(3)

,HS,TR,AG,media
LM-KN,0.833,0.878,0.796,0.835
LM-KN-PS,0.793,0.890,0.770,0.818
LM-KN+lex,0.949,0.972,0.961,0.961
(referencia) CNB,0.843,0.869,0.794,0.835
(referencia) SVM,0.993,0.994,0.993,0.994
(referencia) NB-LR,0.965,0.982,0.972,0.973
(referencia) Powerset,0.974,0.988,0.980,0.981
(referencia) LGBM,0.909,0.962,0.928,0.933


## 3. Stacking
Primero el stacking simple (una semilla de partición por base, como E0) con los 6 modelos de E8b (B6 = 5 originales + LightGBM). Después el candidato real: `stacking_bagged` con 5 semillas de partición (42-46) por base, el mismo protocolo de E8b, evaluado con cada semilla del metamodelo en `tb.SEEDS`.

In [5]:
B6 = tb.BASES_ORIGINALES + ['LGBM']
S5 = (42, 43, 44, 45, 46)
R2 = {}
for k, extra in {'B6': [], 'B6 + LM-KN': ['LM-KN'], 'B6 + LM-KN-PS': ['LM-KN-PS'], 'B6 + LM-KN+lex': ['LM-KN+lex']}.items():
    R2['Stacking simple · ' + k] = tb.evaluar(d, {s: tb.stacking(d, B6 + extra, s, con_test=False)[0] for s in tb.SEEDS})
for k, extra in {'B6 (E8b)': [], 'B6 + LM-KN': ['LM-KN'], 'B6 + LM-KN+lex': ['LM-KN+lex']}.items():
    R2['Bagging 5 semillas · ' + k] = tb.evaluar(d, {s: tb.stacking_bagged(d, B6 + extra, S5, s, con_test=False)[0] for s in tb.SEEDS})
tb.log('Stacking listo')
tb.tabla(R2)

[     4s] Stacking listo


,F1 HS,F1 TR,F1 AG,F1 medio,±,EMR,F1 medio por semilla
Configuración,,,,,,,
Stacking simple · B6,0.8072,0.8336,0.7982,0.8130,0.0013,0.7737,0.814 / 0.813 / 0.811
Stacking simple · B6 + LM-KN,0.8072,0.8337,0.7975,0.8128,0.0014,0.7726,0.812 / 0.815 / 0.811
Stacking simple · B6 + LM-KN-PS,0.8090,0.8346,0.7959,0.8132,0.0014,0.7737,0.813 / 0.815 / 0.812
Stacking simple · B6 + LM-KN+lex,0.8089,0.8335,0.7997,0.8140,0.0017,0.7743,0.815 / 0.812 / 0.816
Bagging 5 semillas · B6 (E8b),0.8071,0.8405,0.8062,0.8179,0.0012,0.7785,0.819 / 0.818 / 0.816
Bagging 5 semillas · B6 + LM-KN,0.8113,0.8376,0.7997,0.8162,0.0007,0.7753,0.817 / 0.816 / 0.816
Bagging 5 semillas · B6 + LM-KN+lex,0.8106,0.8397,0.8066,0.8189,0.0008,0.7795,0.818 / 0.820 / 0.819


In [6]:
# Diferencia por semilla del metamodelo frente a E8b (criterio de adopción: >= +0,003 y mejora en 2 de 3 semillas)
base = {s: tb.metricas(d.Y, tb.binarizar(tb.stacking_bagged(d, B6, S5, s, con_test=False)[0]))['F1 medio'] for s in tb.SEEDS}
filas = []
for extra in (['LM-KN'], ['LM-KN+lex']):
    f = {s: tb.metricas(d.Y, tb.binarizar(tb.stacking_bagged(d, B6 + extra, S5, s, con_test=False)[0]))['F1 medio'] for s in tb.SEEDS}
    filas.append({'Modelo': 'B6 + ' + extra[0], **{f'Δ s{s}': f[s] - base[s] for s in tb.SEEDS},
                  'Δ media': np.mean([f[s] - base[s] for s in tb.SEEDS])})
# Caché disponible para un envío futuro (OOF y predicciones de test por partición de las 5 semillas)
cache = {m: all(os.path.exists(tb._ruta(d, m, tb._cfg(d, None), suf)) for s in S5 for suf in (f's{s}', f'testfolds_s{s}'))
         for m in ('LM-KN', 'LM-KN+lex')}
print('OOF + testfolds de las 5 semillas en caché:', cache)
pd.DataFrame(filas).set_index('Modelo').round(4)

OOF + testfolds de las 5 semillas en caché: {'LM-KN': True, 'LM-KN+lex': True}


,Δ s42,Δ s43,Δ s44,Δ media
Modelo,,,,
B6 + LM-KN,-0.0018,-0.0029,-0.0004,-0.0017
B6 + LM-KN+lex,-0.0010,0.0015,0.0026,0.0010


## Conclusión: **no se adopta**
- **El LM por clase solo es débil:** `LM-KN` llega a un F1 medio de **0,758 ± 0,001**, 4,4 puntos por debajo de la LR (0,802). El powerset generativo (`LM-KN-PS`) es peor (0,732): las combinaciones raras (`110`, `001`, con 20-30 tweets) dan LMs casi vacíos. Los LMs de caracteres aportan poco frente a los de palabras (0,716 frente a 0,752 con la semilla 42): con ~700 tweets por clase, la probabilidad de un tweet la domina el español común, no la clase.
- **Sí es una vista distinta:** la correlación de sus OOF con la LR es 0,835 de media (0,82 el powerset), al nivel de CNB y muy por debajo de SVM, NB-LR o Powerset (0,97-0,99). La inspección de la sección 1 muestra el problema: el LM reconoce bien *callate zorra* o *malditos sudacas*, pero también castiga como "no odio" textos con *puta boca* o *arabe* que el anotador marcó como 0 por razones de contexto, y premia patrones superficiales (*xuser jajaja ...*).
- **Pero esa diversidad no se convierte en F1:** con el protocolo de E8b (*bagging* de 5 semillas), agregar `LM-KN` **empeora** el stacking en las 3 semillas (0,8162 frente a 0,8179; Δ = −0,0017). `LM-KN+lex` (LLR + léxico) da 0,8189 (Δ = +0,0010; mejora en 2 de 3 semillas, −0,0010 / +0,0015 / +0,0026), por debajo del criterio de adopción (≥ +0,003) y dentro de la meseta de E9 (0,8164-0,8184). Además, su aporte puede venir del léxico y no del LM: `LM-KN+lex` correlaciona 0,96 con la LR.
- **Lectura:** coincide con la conclusión general de `RESUMEN.md`. La información útil está en el contexto fino (*hija de puta* frente a *hijo de puta*, *puta* como intensificador), y un modelo de n-gramas por clase con ~1.400 tweets por partición no la estima mejor que los n-gramas TF-IDF que ya tiene el stacking; lo que sí mide (el estilo general del tweet) no ayuda a separar las clases.
- **Lo que queda en caché** (`oof/base/`): OOF y predicciones de test por partición (`testfolds`) de las 5 semillas (42-46) para `LM-KN` y `LM-KN+lex`, y OOF de las semillas 42-44 para `LM-KN-PS`. Para reutilizarlos: `import ext_lm` después de `import toxic_base as tb`. No se generó ninguna submission.